# Data Collection

## Set-Up

In [59]:
import requests
import pandas as pd
import numpy as np
from pprint import pprint
import wikipedia as wp
import io
import json
from datetime import datetime

## Venues

In [60]:
league_venues_dict = {'nfl': {'page_title': 'List of current NFL stadiums', 'table_number': 1, 'cols_to_remove': ['Image', 'Ref(s)']},
           'mls': {'page_title': 'List of Major League Soccer stadiums', 'table_number': 1, 'cols_to_remove': ['Image', 'Ref(s)', 'First MLS year in stadium', 'Field dimensions', 'Coordinates']},
           'nba': {'page_title': 'List of NBA arenas', 'table_number': 0, 'cols_to_remove': ['Image', 'Ref(s)', 'Season of first NBA game']},
           'mlb': {'page_title': 'List of current Major League Baseball stadiums', 'table_number': 0, 'cols_to_remove': ['Image', 'Distance to center field']},
           'nhl': {'page_title': 'List of National Hockey League arenas', 'table_number': 0, 'cols_to_remove': ['Image', 'Ref(s)', 'Season of first NHL game']}
           }

def wp_table(league, leagues_dict=league_venues_dict):
    # retrieve wiki page
    wp_page = wp.page(leagues_dict[league]['page_title'])

    # get raw html, convert to StringIO to satisfy pd.read_html()
    html = io.StringIO(wp_page.html())

    # retrieve target table, remove unneeded columns
    df = pd.read_html(html)[leagues_dict[league]['table_number']].drop(columns=leagues_dict[league]['cols_to_remove'])

    # remove ASCII characters
    df['Capacity'] = df['Capacity'].astype('str')
    df = df.apply(lambda col: col.str.encode('ascii', 'ignore').str.decode('ascii'))

    # filter out [references]
    df = df.apply(lambda col: col.str.replace(r'\[.+\]|\*', '', regex=True))

    # remove commas and parens from Capacity, convert to Int
    df['Capacity'] = df['Capacity'].str.replace(r'\(.+\)|,', '', regex=True)
    df['Capacity'] = df['Capacity'].astype(int)

    # add league column
    df['League'] = league

    # league-specific needs
    if league=='nfl':
        df['Teams'] = df['Team(s)'].apply(lambda x: x.split(' ') if x.count(' ') > 2 else x)
        # df['Team 1'] = [x[0] + ' ' + x[1] + ' ' + x[2] if type(x) is list else x for x in df['Teams']]
        # df['Team 2'] = [x[3] + ' ' + x[4] + ' ' + x[5] if type(x) is list else None for x in df['Teams']]
        df['Teams'] = [[x[0]+' '+x[1]+' '+ x[2], x[3]+' '+x[4]+' '+x[5]] if type(x) is list else x for x in df['Teams']]
        df = df.explode('Teams')
        df = df.rename(columns={'Teams': 'Team'})
        df = df.drop(columns=['Team(s)'])
    elif league=='mls':
        df['Team'] = df['Team'].str.replace('Montral', 'Montreal')

    return df

In [61]:
# collect data
nfl_venues = wp_table('nfl')
mls_venues = wp_table('mls')
nba_venues = wp_table('nba')
mlb_venues = wp_table('mlb')
nhl_venues = wp_table('nhl')

In [62]:
# preview results
print('NFL')
display(nfl_venues.tail(3))

print('\nMLS')
display(mls_venues.tail(3))

print('\nNBA')
display(nba_venues.tail(3))

print('\nMLB')
display(mlb_venues.tail(3))

print('\nNHL')
display(nhl_venues.tail(3))

NFL


,Name,Location,Capacity,Surface,Roof type,Opened,League,Team
27,Soldier Field,"Chicago, Illinois",61500,Bermuda grass,Open,1924,nfl,Chicago Bears
28,State Farm Stadium,"Glendale, Arizona",63400,Bermuda grass,Retractable,2006,nfl,Arizona Cardinals
29,U.S. Bank Stadium,"Minneapolis, Minnesota",66655,UBU Speed Series S5-M Synthetic Turf,Fixed,2016,nfl,Minnesota Vikings



MLS


,Stadium,Team,Location,Capacity,Opened,Surface,Roof type,League
27,Toyota Stadium,FC Dallas,"Frisco, Texas",19096,2005,Grass,Open,mls
28,TQL Stadium,FC Cincinnati,"Cincinnati, Ohio",26000,2021,Hybrid grass,Open,mls
29,Yankee Stadium,New York City FC,"The Bronx, New York",30321,2009,Grass,Open,mls



NBA


,Arena,Location,Team,Capacity,Opened,League
27,Toyota Center,"Houston, Texas",Houston Rockets,18055,2003,nba
28,United Center,"Chicago, Illinois",Chicago Bulls,20917,1994,nba
29,Xfinity Mobile Arena,"Philadelphia, Pennsylvania",Philadelphia 76ers,20478,1996,nba



MLB


,Name,Capacity,Location,Surface,Team,Opened,Type,Roof type,League
27,Truist Park,41084,"Cumberland, Georgia",Grass,Atlanta Braves,2017,Retro-modern,Open,mlb
28,Wrigley Field,41649,"Chicago, Illinois",Grass,Chicago Cubs,1914,Jewel box,Open,mlb
29,Yankee Stadium,46537,"Bronx, New York",Grass,New York Yankees,2009,Retro-classic,Open,mlb



NHL


,Arena,Location,Team,Capacity,Opened,League
29,UBS Arena,"Elmont, New York",New York Islanders,17255,2021,nhl
30,United Center,"Chicago, Illinois",Chicago Blackhawks,19717,1994,nhl
31,Xfinity Mobile Arena,"Philadelphia, Pennsylvania",Philadelphia Flyers,19538,1996,nhl


In [63]:
venues_list = [nfl_venues, mls_venues, nba_venues, mlb_venues, nhl_venues]
venues_dfs = []

for df in venues_list:
    df = df.rename(columns={'Arena': 'Venue', 'Stadium': 'Venue', 'Name': 'Venue', 'Location': 'City'})
    df = df[['Venue', 'Team', 'City', 'Capacity', 'Opened', 'League']]
    venues_dfs.append(df)
    
all_venues = pd.concat(venues_dfs).sort_values(['League', 'Team']).reset_index(drop=True)
all_venues

,Venue,Team,City,Capacity,Opened,League
0,Chase Field,Arizona Diamondbacks,"Phoenix, Arizona",48330,1998,mlb
1,Sutter Health Park,Athletics,"West Sacramento, California",13416,2000,mlb
2,Truist Park,Atlanta Braves,"Cumberland, Georgia",41084,2017,mlb
3,Oriole Park at Camden Yards,Baltimore Orioles,"Baltimore, Maryland",44970,1992,mlb
4,Fenway Park,Boston Red Sox,"Boston, Massachusetts",37755,1912,mlb
...,...,...,...,...,...,...
149,Delta Center,Utah Mammoth,"Salt Lake City, Utah",16200,1991,nhl
150,Rogers Arena,Vancouver Canucks,"Vancouver, British Columbia",18910,1995,nhl
151,T-Mobile Arena,Vegas Golden Knights,"Paradise, Nevada",17367,2016,nhl
152,Capital One Arena,Washington Capitals,"Washington, D.C.",18573,1997,nhl


In [64]:
# verify team names align between WP venues and team_completions spreadsheet
team_completions = pd.read_excel('etl/data/team_completions.xlsx').sort_values(by=['league','team']).reset_index(drop=True)
team_completions

,league,home_id,team,schedule_name,metro_area,metro_team_num,completed_flag
0,mlb,mlb_1,Arizona Diamondbacks,Arizona Diamondbacks,"Phoenix, Arizona",mlb_1,0
1,mlb,mlb_20,Athletics,Athletics,"Las Vegas, Nevada",mlb_1,0
2,mlb,mlb_2,Atlanta Braves,Atlanta Braves,"Atlanta, Georgia",mlb_1,1
3,mlb,mlb_3,Baltimore Orioles,Baltimore Orioles,"Baltimore, Maryland",mlb_1,0
4,mlb,mlb_4,Boston Red Sox,Boston Red Sox,"Boston, Massachusetts",mlb_1,0
...,...,...,...,...,...,...,...
149,nhl,nhl_28,Utah Mammoth,Utah Hockey Club,"Salt Lake City, Utah",nhl_1,0
150,nhl,nhl_29,Vancouver Canucks,Vancouver Canucks,"Vancouver, British Columbia",nhl_1,0
151,nhl,nhl_30,Vegas Golden Knights,Vegas Golden Knights,"Las Vegas, Nevada",nhl_1,0
152,nhl,nhl_31,Washington Capitals,Washington Capitals,"Washington, D.C.",nhl_1,0


In [65]:
# join teams to venues
team_venues = team_completions.merge(all_venues, how='outer', left_on='team', right_on='Team', indicator=True)
team_venues

,league,home_id,team,schedule_name,metro_area,metro_team_num,completed_flag,Venue,Team,City,Capacity,Opened,League,_merge
0,nhl,nhl_1,Anaheim Ducks,Anaheim Ducks,"Los Angeles, California",nhl_1,0,Honda Center,Anaheim Ducks,"Anaheim, California",17174,1993,nhl,both
1,nfl,nfl_1,Arizona Cardinals,Arizona Cardinals,"Phoenix, Arizona",nfl_1,0,State Farm Stadium,Arizona Cardinals,"Glendale, Arizona",63400,2006,nfl,both
2,mlb,mlb_1,Arizona Diamondbacks,Arizona Diamondbacks,"Phoenix, Arizona",mlb_1,0,Chase Field,Arizona Diamondbacks,"Phoenix, Arizona",48330,1998,mlb,both
3,mlb,mlb_20,Athletics,Athletics,"Las Vegas, Nevada",mlb_1,0,Sutter Health Park,Athletics,"West Sacramento, California",13416,2000,mlb,both
4,mlb,mlb_2,Atlanta Braves,Atlanta Braves,"Atlanta, Georgia",mlb_1,1,Truist Park,Atlanta Braves,"Cumberland, Georgia",41084,2017,mlb,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,nhl,nhl_31,Washington Capitals,Washington Capitals,"Washington, D.C.",nhl_1,0,Capital One Arena,Washington Capitals,"Washington, D.C.",18573,1997,nhl,both
150,nfl,nfl_32,Washington Commanders,Washington Commanders,"Washington, D.C.",nfl_1,0,Northwest Stadium,Washington Commanders,"Landover, Maryland",62000,1997,nfl,both
151,mlb,mlb_30,Washington Nationals,Washington Nationals,"Washington, D.C.",mlb_1,1,Nationals Park,Washington Nationals,"Washington, D.C.",41373,2008,mlb,both
152,nba,nba_30,Washington Wizards,Washington Wizards,"Washington, D.C.",nba_1,0,Capital One Arena,Washington Wizards,"Washington, D.C.",20356,1997,nba,both


In [66]:
# df should be empty
team_venues[team_venues['_merge']!='both']

,league,home_id,team,schedule_name,metro_area,metro_team_num,completed_flag,Venue,Team,City,Capacity,Opened,League,_merge


In [67]:
# uncomment if teams aren't matching up
# use to update the spreadsheet so the team names are current
# all_venues.to_excel('etl/data/venue_teams.xlsx')

In [68]:
team_venues = team_venues[['Venue', 'Team', 'City', 'Capacity', 'Opened']].sort_values(by=['Team']).reset_index(drop=True)
team_venues

,Venue,Team,City,Capacity,Opened
0,Honda Center,Anaheim Ducks,"Anaheim, California",17174,1993
1,State Farm Stadium,Arizona Cardinals,"Glendale, Arizona",63400,2006
2,Chase Field,Arizona Diamondbacks,"Phoenix, Arizona",48330,1998
3,Sutter Health Park,Athletics,"West Sacramento, California",13416,2000
4,Truist Park,Atlanta Braves,"Cumberland, Georgia",41084,2017
...,...,...,...,...,...
149,Capital One Arena,Washington Capitals,"Washington, D.C.",18573,1997
150,Northwest Stadium,Washington Commanders,"Landover, Maryland",62000,1997
151,Nationals Park,Washington Nationals,"Washington, D.C.",41373,2008
152,Capital One Arena,Washington Wizards,"Washington, D.C.",20356,1997


## Games

In [69]:
# TODO: make this dynamic with try-except instead of hard-coding available seasons
league_games_seasons = {'nfl': [2025], 'mls': [2025], 'nhl': [2025], 'mlb': [2025], 'nba': [2025]}

def get_games(league):
    game_dfs = []
    for season in league_games_seasons[league]:
        url = f'https://fixturedownload.com/feed/json/{league}-{season}'
        response = requests.get(url)
        raw_json = response.text
        df = pd.DataFrame(json.loads(raw_json))
        df['DateUtc'] = pd.to_datetime(df['DateUtc'])
        season_start = df['DateUtc'].min().year
        if league in ['nfl', 'nhl', 'nba']:
            df['Season'] = str(season_start) + "-" + str(season_start+1)
        elif league in ['mlb', 'mls']:
            df['Season'] = str(season_start)
        else:
            print('Invalid league name')
            break
        game_dfs.append(df)
    games_df = pd.concat(game_dfs)
    games_df['League'] = league.upper()
    games_df.loc[games_df['League']=='MLS', 'HomeTeam'] = games_df.loc[games_df['League']=='MLS', 'HomeTeam'] + ' MLS'
    games_df.loc[games_df['League']=='MLS', 'AwayTeam'] = games_df.loc[games_df['League']=='MLS', 'AwayTeam'] + ' MLS'
    games_df.reset_index(drop=True, inplace=True)
    return games_df

In [71]:
# collect data
nfl_games = get_games('nfl')
mls_games = get_games('mls')
nhl_games = get_games('nhl')
mlb_games = get_games('mlb')
nba_games = get_games('nba')

In [72]:
# preview results
print('NFL')
display(nfl_games.head(3))
display(nfl_games.tail(3))

print('\nMLS')
display(mls_games.head(3))
display(mls_games.tail(3))

print('\nNBA')
display(nba_games.head(3))
display(nba_games.tail(3))

print('\nMLB')
display(mlb_games.head(3))
display(mlb_games.tail(3))

print('\nNHL')
display(nhl_games.head(3))
display(nhl_games.tail(3))

NFL


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
0,1,1,2025-09-05 00:20:00+00:00,Lincoln Financial Field,Philadelphia Eagles,Dallas Cowboys,None,24.0,20.0,2025-2026,NFL
1,2,1,2025-09-06 00:00:00+00:00,Arena Corinthians,Los Angeles Chargers,Kansas City Chiefs,None,27.0,21.0,2025-2026,NFL
2,3,1,2025-09-07 17:00:00+00:00,Mercedes-Benz Stadium,Atlanta Falcons,Tampa Bay Buccaneers,None,NaN,NaN,2025-2026,NFL


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
269,270,18,2026-01-05 00:00:00+00:00,Acrisure Stadium,Pittsburgh Steelers,Baltimore Ravens,None,NaN,NaN,2025-2026,NFL
270,271,18,2026-01-05 00:00:00+00:00,Levi's Stadium,San Francisco 49ers,Seattle Seahawks,None,NaN,NaN,2025-2026,NFL
271,272,18,2026-01-05 00:00:00+00:00,Raymond James Stadium,Tampa Bay Buccaneers,Carolina Panthers,None,NaN,NaN,2025-2026,NFL



MLS


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
0,2,1,2025-02-22 21:45:00+00:00,BMO Stadium,LAFC MLS,Minnesota MLS,None,1.0,0.0,2025,MLS
1,1,1,2025-02-23 00:30:00+00:00,Chase Stadium,Miami MLS,New York City MLS,None,2.0,2.0,2025,MLS
2,3,1,2025-02-23 00:30:00+00:00,Mercedes-Benz Stadium,Atlanta MLS,Montréal MLS,None,3.0,2.0,2025,MLS


...

,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
0,1,1,2025-03-18 10:10:00+00:00,Tokyo Dome,Chicago Cubs,Los Angeles Dodgers,None,1.0,4.0,2025,MLB
1,2,1,2025-03-19 10:10:00+00:00,Tokyo Dome,Chicago Cubs,Los Angeles Dodgers,None,3.0,6.0,2025,MLB
2,3,1,2025-03-27 19:05:00+00:00,Yankee Stadium,New York Yankees,Milwaukee Brewers,None,4.0,2.0,2025,MLB


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
2427,2428,27,2025-09-28 19:10:00+00:00,American Family Field,Milwaukee Brewers,Cincinnati Reds,None,NaN,NaN,2025,MLB
2428,2429,27,2025-09-28 19:15:00+00:00,Truist Park,Atlanta Braves,Pittsburgh Pirates,None,NaN,NaN,2025,MLB
2429,2430,27,2025-09-28 19:20:00+00:00,Wrigley Field,Chicago Cubs,St. Louis Cardinals,None,NaN,NaN,2025,MLB



NHL


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
0,1,1,2025-10-07 21:00:00+00:00,Amerant Bank Arena,Florida Panthers,Chicago Blackhawks,None,None,None,2025-2026,NHL
1,2,1,2025-10-08 00:00:00+00:00,Madison Square Garden,New York Rangers,Pittsburgh Penguins,None,None,None,2025-2026,NHL
2,3,1,2025-10-08 02:30:00+00:00,Crypto.com Arena,Los Angeles Kings,Colorado Avalanche,None,None,None,2025-2026,NHL


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League
1309,1310,24,2026-04-17 01:00:00+00:00,Scotiabank Saddledome,Calgary Flames,Los Angeles Kings,None,None,None,2025-2026,NHL
1310,1311,24,2026-04-17 01:00:00+00:00,Rogers Place,Edmonton Oilers,Vancouver Canucks,None,None,None,2025-2026,NHL
1311,1312,24,2026-04-17 02:30:00+00:00,Ball Arena,Colorado Avalanche,Seattle Kraken,None,None,None,2025-2026,NHL


In [73]:
all_games = pd.concat([nfl_games, mls_games, nba_games, mlb_games, nhl_games])
all_games['GameId'] = all_games['League'] + '_' + all_games['Season'].str[:4] + '_' + all_games['MatchNumber'].astype(str)
all_games

C:\Users\hunte\AppData\Local\Temp\ipykernel_32128\841450259.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_games = pd.concat([nfl_games, mls_games, nba_games, mlb_games, nhl_games])


,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,League,GameId
0,1,1,2025-09-05 00:20:00+00:00,Lincoln Financial Field,Philadelphia Eagles,Dallas Cowboys,None,24.0,20.0,2025-2026,NFL,NFL_2025_1
1,2,1,2025-09-06 00:00:00+00:00,Arena Corinthians,Los Angeles Chargers,Kansas City Chiefs,None,27.0,21.0,2025-2026,NFL,NFL_2025_2
2,3,1,2025-09-07 17:00:00+00:00,Mercedes-Benz Stadium,Atlanta Falcons,Tampa Bay Buccaneers,None,NaN,NaN,2025-2026,NFL,NFL_2025_3
3,4,1,2025-09-07 17:00:00+00:00,Huntington Bank Field,Cleveland Browns,Cincinnati Bengals,None,NaN,NaN,2025-2026,NFL,NFL_2025_4
4,5,1,2025-09-07 17:00:00+00:00,Lucas Oil Stadium,Indianapolis Colts,Miami Dolphins,None,NaN,NaN,2025-2026,NFL,NFL_2025_5
...,...,...,...,...,...,...,...,...,...,...,...,...
1307,1308,24,2026-04-17 00:00:00+00:00,Canada Life Centre,Winnipeg Jets,San Jose Sharks,None,NaN,NaN,2025-2026,NHL,NHL_2025_1308
1308,1309,24,2026-04-17 00:00:00+00:00,Delta Center,Utah Hockey Club,St. Louis Blues,None,NaN,NaN,2025-2026,NHL,NHL_2025_1309
1309,1310,24,2026-04-17 01:00:00+00:00,Scotiabank Saddledome,Calgary Flames,Los Angeles Kings,None,NaN,NaN,2025-2026,NHL,NHL_2025_1310
1310,1311,24,2026-04-17 01:00:00+00:00,Rogers Place,Edmonton Oilers,Vancouver Canucks,None,NaN,NaN,2025-2026,NHL,NHL_2025_1311


## Teams

In [74]:
leagues_teams_games = {'nfl': nfl_games, 'mls': mls_games, 'nba': nba_games, 'mlb': mlb_games, 'nhl': nhl_games}
def get_teams(league):
    teams_df = leagues_teams_games[league]['HomeTeam'].drop_duplicates().sort_values().reset_index(drop=True)
    teams_df = teams_df.reset_index().rename(columns={'index': 'TeamId', 'HomeTeam': 'TeamName'})
    teams_df['TeamId'] = teams_df['TeamId'] + 1
    teams_df['League'] = league
    teams_df['TeamId'] = teams_df['League'] + '_' + teams_df['TeamId'].astype(str)
    return teams_df

In [75]:
# collect data
nfl_teams = get_teams('nfl')
mls_teams = get_teams('mls')
nhl_teams = get_teams('nhl')
mlb_teams = get_teams('mlb')
nba_teams = get_teams('nba')

In [76]:
# preview results
print('NFL')
display(nfl_teams.head(3))
display(nfl_teams.tail(3))

print('\nMLS')
display(mls_teams.head(3))
display(mls_teams.tail(3))

print('\nNBA')
display(nba_teams.head(3))
display(nba_teams.tail(3))

print('\nMLB')
display(mlb_teams.head(3))
display(mlb_teams.tail(3))

print('\nNHL')
display(nhl_teams.head(3))
display(nhl_teams.tail(3))

NFL


,TeamId,TeamName,League
0,nfl_1,Arizona Cardinals,nfl
1,nfl_2,Atlanta Falcons,nfl
2,nfl_3,Baltimore Ravens,nfl


,TeamId,TeamName,League
29,nfl_30,Tampa Bay Buccaneers,nfl
30,nfl_31,Tennessee Titans,nfl
31,nfl_32,Washington Commanders,nfl



MLS


,TeamId,TeamName,League
0,mls_1,Atlanta MLS,mls
1,mls_2,Austin MLS,mls
2,mls_3,Charlotte MLS,mls


...

,TeamId,TeamName,League
0,mlb_1,Arizona Diamondbacks,mlb
1,mlb_2,Athletics,mlb
2,mlb_3,Atlanta Braves,mlb


,TeamId,TeamName,League
27,mlb_28,Texas Rangers,mlb
28,mlb_29,Toronto Blue Jays,mlb
29,mlb_30,Washington Nationals,mlb



NHL


,TeamId,TeamName,League
0,nhl_1,Anaheim Ducks,nhl
1,nhl_2,Boston Bruins,nhl
2,nhl_3,Buffalo Sabres,nhl


,TeamId,TeamName,League
29,nhl_30,Vegas Golden Knights,nhl
30,nhl_31,Washington Capitals,nhl
31,nhl_32,Winnipeg Jets,nhl


In [77]:
all_teams = pd.concat([nfl_teams, mls_teams, nhl_teams, mlb_teams, nba_teams]).sort_values(by=['League','TeamName']).reset_index(drop=True)
print(f"# of teams: {len(all_teams)}")
display(all_teams)

# of teams: 154


,TeamId,TeamName,League
0,mlb_1,Arizona Diamondbacks,mlb
1,mlb_2,Athletics,mlb
2,mlb_3,Atlanta Braves,mlb
3,mlb_4,Baltimore Orioles,mlb
4,mlb_5,Boston Red Sox,mlb
...,...,...,...
149,nhl_28,Utah Hockey Club,nhl
150,nhl_29,Vancouver Canucks,nhl
151,nhl_30,Vegas Golden Knights,nhl
152,nhl_31,Washington Capitals,nhl


In [78]:
teams_dfs = all_teams.merge(team_completions, how='outer', left_on='TeamName', right_on='schedule_name', indicator=True)
teams_dfs

,TeamId,TeamName,League,league,home_id,team,schedule_name,metro_area,metro_team_num,completed_flag,_merge
0,nhl_1,Anaheim Ducks,nhl,nhl,nhl_1,Anaheim Ducks,Anaheim Ducks,"Los Angeles, California",nhl_1,0,both
1,nfl_1,Arizona Cardinals,nfl,nfl,nfl_1,Arizona Cardinals,Arizona Cardinals,"Phoenix, Arizona",nfl_1,0,both
2,mlb_1,Arizona Diamondbacks,mlb,mlb,mlb_1,Arizona Diamondbacks,Arizona Diamondbacks,"Phoenix, Arizona",mlb_1,0,both
3,mlb_2,Athletics,mlb,mlb,mlb_20,Athletics,Athletics,"Las Vegas, Nevada",mlb_1,0,both
4,mlb_3,Atlanta Braves,mlb,mlb,mlb_2,Atlanta Braves,Atlanta Braves,"Atlanta, Georgia",mlb_1,1,both
...,...,...,...,...,...,...,...,...,...,...,...
149,nhl_31,Washington Capitals,nhl,nhl,nhl_31,Washington Capitals,Washington Capitals,"Washington, D.C.",nhl_1,0,both
150,nfl_32,Washington Commanders,nfl,nfl,nfl_32,Washington Commanders,Washington Commanders,"Washington, D.C.",nfl_1,0,both
151,mlb_30,Washington Nationals,mlb,mlb,mlb_30,Washington Nationals,Washington Nationals,"Washington, D.C.",mlb_1,1,both
152,nba_30,Washington Wizards,nba,nba,nba_30,Washington Wizards,Washington Wizards,"Washington, D.C.",nba_1,0,both


In [79]:
# df should be empty
teams_dfs[teams_dfs['_merge']!='both']

,TeamId,TeamName,League,league,home_id,team,schedule_name,metro_area,metro_team_num,completed_flag,_merge


In [80]:
# uncomment if teams aren't matching up
# use to update the spreadsheet so the team names are current
# all_teams.to_excel('etl/data/schedule_teams.xlsx')

In [81]:
# subset columns
teams_dfs = teams_dfs[['team', 'schedule_name', 'home_id', 'league', 'metro_area', 'metro_team_num', 'completed_flag']].sort_values(by=['league', 'team']).reset_index(drop=True)
teams_dfs

,team,schedule_name,home_id,league,metro_area,metro_team_num,completed_flag
0,Arizona Diamondbacks,Arizona Diamondbacks,mlb_1,mlb,"Phoenix, Arizona",mlb_1,0
1,Athletics,Athletics,mlb_20,mlb,"Las Vegas, Nevada",mlb_1,0
2,Atlanta Braves,Atlanta Braves,mlb_2,mlb,"Atlanta, Georgia",mlb_1,1
3,Baltimore Orioles,Baltimore Orioles,mlb_3,mlb,"Baltimore, Maryland",mlb_1,0
4,Boston Red Sox,Boston Red Sox,mlb_4,mlb,"Boston, Massachusetts",mlb_1,0
...,...,...,...,...,...,...,...
149,Utah Mammoth,Utah Hockey Club,nhl_28,nhl,"Salt Lake City, Utah",nhl_1,0
150,Vancouver Canucks,Vancouver Canucks,nhl_29,nhl,"Vancouver, British Columbia",nhl_1,0
151,Vegas Golden Knights,Vegas Golden Knights,nhl_30,nhl,"Las Vegas, Nevada",nhl_1,0
152,Washington Capitals,Washington Capitals,nhl_31,nhl,"Washington, D.C.",nhl_1,0


## Combining venues, teams, and games

In [82]:
# join back against all games df
all_games_teams = all_games.merge(teams_dfs, left_on='HomeTeam', right_on='schedule_name')
all_games_teams = all_games_teams.merge(teams_dfs, left_on='AwayTeam', right_on='schedule_name', suffixes=['_home', '_away'])
all_games_venues = all_games_teams.merge(team_venues, left_on='team_home', right_on='Team')
all_games_venues

,MatchNumber,RoundNumber,DateUtc,Location,HomeTeam,AwayTeam,Group,HomeTeamScore,AwayTeamScore,Season,...,home_id_away,league_away,metro_area_away,metro_team_num_away,completed_flag_away,Venue,Team,City,Capacity,Opened
0,1,1,2025-09-05 00:20:00+00:00,Lincoln Financial Field,Philadelphia Eagles,Dallas Cowboys,None,24.0,20.0,2025-2026,...,nfl_9,nfl,"Dallas, Texas",nfl_1,0,Lincoln Financial Field,Philadelphia Eagles,"Philadelphia, Pennsylvania",69596,2003
1,2,1,2025-09-06 00:00:00+00:00,Arena Corinthians,Los Angeles Chargers,Kansas City Chiefs,None,27.0,21.0,2025-2026,...,nfl_16,nfl,"Kansas City, Missouri",nfl_1,1,SoFi Stadium,Los Angeles Chargers,"Inglewood, California",70240,2020
2,3,1,2025-09-07 17:00:00+00:00,Mercedes-Benz Stadium,Atlanta Falcons,Tampa Bay Buccaneers,None,NaN,NaN,2025-2026,...,nfl_30,nfl,"Tampa, Florida",nfl_1,0,Mercedes-Benz Stadium,Atlanta Falcons,"Atlanta, Georgia",71000,2017
3,4,1,2025-09-07 17:00:00+00:00,Huntington Bank Field,Cleveland Browns,Cincinnati Bengals,None,NaN,NaN,2025-2026,...,nfl_7,nfl,"Cincinnati, Ohio",nfl_1,0,Huntington Bank Field,Cleveland Browns,"Cleveland, Ohio",67431,1999
4,5,1,2025-09-07 17:00:00+00:00,Lucas Oil Stadium,Indianapolis Colts,Miami Dolphins,None,NaN,NaN,2025-2026,...,nfl_20,nfl,"Miami, Florida",nfl_1,0,Lucas Oil Stadium,Indianapolis Colts,"Indianapolis, Indiana",67000,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5719,1308,24,2026-04-17 00:00:00+00:00,Canada Life Centre,Winnipeg Jets,San Jose Sharks,None,NaN,NaN,2025-2026,...,nhl_23,nhl,"San Francisco, California",nhl_1,0,Canada Life Centre,Winnipeg Jets,"Winnipeg, Manitoba",15321,2004
5720,1309,24,2026-04-17 00:00:00+00:00,Delta Center,Utah Hockey Club,St. Louis Blues,None,NaN,NaN,2025-2026,...,nhl_25,nhl,"St. Louis, Missouri",nhl_1,1,Delta Center,Utah Mammoth,"Salt Lake City, Utah",16200,1991
5721,1310,24,2026-04-17 01:00:00+00:00,Scotiabank Saddledome,Calgary Flames,Los Angeles Kings,None,NaN,NaN,2025-2026,...,nhl_13,nhl,"Los Angeles, California",nhl_2,0,Scotiabank Saddledome,Calgary Flames,"Calgary, Alberta",19289,1983
5722,1311,24,2026-04-17 01:00:00+00:00,Rogers Place,Edmonton Oilers,Vancouver Canucks,None,NaN,NaN,2025-2026,...,nhl_29,nhl,"Vancouver, British Columbia",nhl_1,0,Rogers Place,Edmonton Oilers,"Edmonton, Alberta",18347,2016


In [83]:
# subset columns
games_df = all_games_venues[['GameId', 'DateUtc', 'team_home', 'home_id_home', 'team_away', 'home_id_away', 'Location', 'HomeTeamScore', 'AwayTeamScore', 'league_home', 'metro_area_home',
                             'metro_team_num_home', 'completed_flag_home', 'Venue', 'City', 'Capacity', 'Opened']]
games_df.columns = ['game_id', 'date', 'home_team', 'home_team_id', 'away_team', 'away_team_id', 'location', 'home_score', 'away_score', 'league',
                    'metro_area', 'metro_team_num', 'completed_flag', 'home_venue', 'venue_city', 'venue_capacity', 'venue_opened']

# format game location and home_venue
games_df.loc[:,'location'] = games_df['location'].str.strip()
games_df.loc[:,'home_venue'] = games_df['home_venue'].str.strip()

# format date to Central Time
# games_df.loc[:,'date'] = pd.to_datetime(games_df['date'])
# games_df['date'] = games_df['date'].dt.tz_convert(tz='US/Central')

# view games df
games_df

,game_id,date,home_team,home_team_id,away_team,away_team_id,location,home_score,away_score,league,metro_area,metro_team_num,completed_flag,home_venue,venue_city,venue_capacity,venue_opened
0,NFL_2025_1,2025-09-05 00:20:00+00:00,Philadelphia Eagles,nfl_26,Dallas Cowboys,nfl_9,Lincoln Financial Field,24.0,20.0,nfl,"Philadelphia, Pennsylvania",nfl_1,0,Lincoln Financial Field,"Philadelphia, Pennsylvania",69596,2003
1,NFL_2025_2,2025-09-06 00:00:00+00:00,Los Angeles Chargers,nfl_18,Kansas City Chiefs,nfl_16,Arena Corinthians,27.0,21.0,nfl,"Los Angeles, California",nfl_1,0,SoFi Stadium,"Inglewood, California",70240,2020
2,NFL_2025_3,2025-09-07 17:00:00+00:00,Atlanta Falcons,nfl_2,Tampa Bay Buccaneers,nfl_30,Mercedes-Benz Stadium,NaN,NaN,nfl,"Atlanta, Georgia",nfl_1,1,Mercedes-Benz Stadium,"Atlanta, Georgia",71000,2017
3,NFL_2025_4,2025-09-07 17:00:00+00:00,Cleveland Browns,nfl_8,Cincinnati Bengals,nfl_7,Huntington Bank Field,NaN,NaN,nfl,"Cleveland, Ohio",nfl_1,0,Huntington Bank Field,"Cleveland, Ohio",67431,1999
4,NFL_2025_5,2025-09-07 17:00:00+00:00,Indianapolis Colts,nfl_14,Miami Dolphins,nfl_20,Lucas Oil Stadium,NaN,NaN,nfl,"Indianapolis, Indiana",nfl_1,0,Lucas Oil Stadium,"Indianapolis, Indiana",67000,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5719,NHL_2025_1308,2026-04-17 00:00:00+00:00,Winnipeg Jets,nhl_32,San Jose Sharks,nhl_23,Canada Life Centre,NaN,NaN,nhl,"Winnipeg, Manitoba",nhl_1,0,Canada Life Centre,"Winnipeg, Manitoba",15321,2004
5720,NHL_2025_1309,2026-04-17 00:00:00+00:00,Utah Mammoth,nhl_28,St. Louis Blues,nhl_25,Delta Center,NaN,NaN,nhl,"Salt Lake City, Utah",nhl_1,0,Delta Center,"Salt Lake City, Utah",16200,1991
5721,NHL_2025_1310,2026-04-17 01:00:00+00:00,Calgary Flames,nhl_4,Los Angeles Kings,nhl_13,Scotiabank Saddledome,NaN,NaN,nhl,"Calgary, Alberta",nhl_1,0,Scotiabank Saddledome,"Calgary, Alberta",19289,1983
5722,NHL_2025_1311,2026-04-17 01:00:00+00:00,Edmonton Oilers,nhl_11,Vancouver Canucks,nhl_29,Rogers Place,NaN,NaN,nhl,"Edmonton, Alberta",nhl_1,0,Rogers Place,"Edmonton, Alberta",18347,2016


In [84]:
# find games where location and home venue don't match
# verify that names match between WP venus and games data
loc = games_df[games_df['location'].str.lower() != games_df['home_venue'].str.lower()]
loc['location'].value_counts()

location
Centre Bell                        41
SAP Center at San Jose             41
Amalie Arena                       40
Stade Saputo                       17
GEHA Field at Arrowhead Stadium     9
Citi Field                          4
SeatGeek Stadium                    3
Tottenham Hotspur Stadium           2
Avicii Arena                        2
Tokyo Dome                          2
Moody Center                        2
Arena Corinthians                   1
The O2 Arena                        1
LoanDepot Park                      1
Journey Bank Ballpark               1
Bristol Motor Speedway              1
TBC                                 1
Uber Arena                          1
Arena CDMX                          1
Levi's Stadium                      1
Stanford Stadium                    1
Huntington Bank Field               1
Santiago Bernabeu Stadium           1
Olympic Stadium                     1
Wembley Stadium                     1
Croke Park                          1
Ray

In [85]:
# update venue names for games data to match WP venues
games_df.loc[:, 'location'] = games_df.loc[:, 'location'].replace({'loanDepot park': 'LoanDepot Park', 'Centre Bell': 'Bell Centre', 'SAP Center at San Jose': 'SAP Center', 
                                                                   'Amalie Arena': 'Benchmark International Arena', 'GEHA Field at Arrowhead Stadium': 'Arrowhead Stadium',
                                                                   'Xcel Energy Center': 'Grand Casino Arena', 'Stade Saputo': 'Saputo Stadium'})
games_df

,game_id,date,home_team,home_team_id,away_team,away_team_id,location,home_score,away_score,league,metro_area,metro_team_num,completed_flag,home_venue,venue_city,venue_capacity,venue_opened
0,NFL_2025_1,2025-09-05 00:20:00+00:00,Philadelphia Eagles,nfl_26,Dallas Cowboys,nfl_9,Lincoln Financial Field,24.0,20.0,nfl,"Philadelphia, Pennsylvania",nfl_1,0,Lincoln Financial Field,"Philadelphia, Pennsylvania",69596,2003
1,NFL_2025_2,2025-09-06 00:00:00+00:00,Los Angeles Chargers,nfl_18,Kansas City Chiefs,nfl_16,Arena Corinthians,27.0,21.0,nfl,"Los Angeles, California",nfl_1,0,SoFi Stadium,"Inglewood, California",70240,2020
2,NFL_2025_3,2025-09-07 17:00:00+00:00,Atlanta Falcons,nfl_2,Tampa Bay Buccaneers,nfl_30,Mercedes-Benz Stadium,NaN,NaN,nfl,"Atlanta, Georgia",nfl_1,1,Mercedes-Benz Stadium,"Atlanta, Georgia",71000,2017
3,NFL_2025_4,2025-09-07 17:00:00+00:00,Cleveland Browns,nfl_8,Cincinnati Bengals,nfl_7,Huntington Bank Field,NaN,NaN,nfl,"Cleveland, Ohio",nfl_1,0,Huntington Bank Field,"Cleveland, Ohio",67431,1999
4,NFL_2025_5,2025-09-07 17:00:00+00:00,Indianapolis Colts,nfl_14,Miami Dolphins,nfl_20,Lucas Oil Stadium,NaN,NaN,nfl,"Indianapolis, Indiana",nfl_1,0,Lucas Oil Stadium,"Indianapolis, Indiana",67000,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5719,NHL_2025_1308,2026-04-17 00:00:00+00:00,Winnipeg Jets,nhl_32,San Jose Sharks,nhl_23,Canada Life Centre,NaN,NaN,nhl,"Winnipeg, Manitoba",nhl_1,0,Canada Life Centre,"Winnipeg, Manitoba",15321,2004
5720,NHL_2025_1309,2026-04-17 00:00:00+00:00,Utah Mammoth,nhl_28,St. Louis Blues,nhl_25,Delta Center,NaN,NaN,nhl,"Salt Lake City, Utah",nhl_1,0,Delta Center,"Salt Lake City, Utah",16200,1991
5721,NHL_2025_1310,2026-04-17 01:00:00+00:00,Calgary Flames,nhl_4,Los Angeles Kings,nhl_13,Scotiabank Saddledome,NaN,NaN,nhl,"Calgary, Alberta",nhl_1,0,Scotiabank Saddledome,"Calgary, Alberta",19289,1983
5722,NHL_2025_1311,2026-04-17 01:00:00+00:00,Edmonton Oilers,nhl_11,Vancouver Canucks,nhl_29,Rogers Place,NaN,NaN,nhl,"Edmonton, Alberta",nhl_1,0,Rogers Place,"Edmonton, Alberta",18347,2016


In [86]:
# find games where location and home venue don't match
# verify that names match between WP venus and games data
loc = games_df[games_df['location'].str.lower() != games_df['home_venue'].str.lower()]
loc['location'].value_counts()

location
Citi Field                   4
SeatGeek Stadium             3
Tottenham Hotspur Stadium    2
Avicii Arena                 2
Tokyo Dome                   2
Moody Center                 2
Arena Corinthians            1
Uber Arena                   1
LoanDepot Park               1
Journey Bank Ballpark        1
Bristol Motor Speedway       1
The O2 Arena                 1
TBC                          1
Arena CDMX                   1
Croke Park                   1
Levi's Stadium               1
Stanford Stadium             1
Huntington Bank Field        1
Santiago Bernabeu Stadium    1
Olympic Stadium              1
Wembley Stadium              1
Raymond James Stadium        1
Name: count, dtype: int64

In [87]:
# export games
games_df.to_csv('etl/games.csv', index=False)